# Weekly Product Demand Forecast

We will forecast the number of units sold for each product at each branch, one week at a time. This can help a shop start a restocking conversation, but it is not an automatic order recommendation. The sales data is synthetic.

## The plan

- **Baseline:** predict that this week will match last week's unit sales.
- **Forecast model:** predict this week using the average units sold in the previous four weeks.
- **Time split:** use the earlier 80% of weeks as history and reserve the latest 20% for validation. We do not shuffle weeks.

We make a one-week-ahead forecast. Once each validation week has finished, its actual sales become available to forecast the next week.

## 1. Load the prepared sales

Run the data_preparation notebook first so the cleaned CSV exists.

In [6]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'data' / 'processed' / 'clean_sales_2025.csv').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

SALES_FILE = ROOT / 'data' / 'processed' / 'clean_sales_2025.csv'
if not SALES_FILE.exists():
    raise FileNotFoundError('Run the Milestone 2 notebook first to create the cleaned sales file.')

sales = pd.read_csv(SALES_FILE, parse_dates=['sale_date'])
REPORTS_DIR = ROOT / 'reports'
print(f'Product lines loaded: {len(sales):,}')
print(f'Sales dates: {sales.sale_date.min():%d %b %Y} to {sales.sale_date.max():%d %b %Y}')

Product lines loaded: 1,511
Sales dates: 01 Jan 2025 to 31 Dec 2025


## 2. Total units by product, branch, and week

The source file only lists weeks when a product sold. We add zero-sales weeks too, so a quiet week is counted as zero rather than disappearing from the forecast.

In [7]:
group_columns = ['store_id', 'store_name', 'product_id', 'product_name']

# Start each week on Monday.
sales['week_start'] = sales['sale_date'] - pd.to_timedelta(sales['sale_date'].dt.weekday, unit='D')
weekly_totals = (
    sales.groupby(group_columns + ['week_start'], as_index=False)
    .agg(units_sold=('quantity', 'sum'))
)

all_weeks = pd.date_range(weekly_totals['week_start'].min(), weekly_totals['week_start'].max(), freq='7D')
week_table = pd.DataFrame({'week_start': all_weeks})
product_branches = weekly_totals[group_columns].drop_duplicates()

# Make every product-branch combination have a row for every week.
weekly = product_branches.merge(week_table, how='cross')
weekly = weekly.merge(weekly_totals, on=group_columns + ['week_start'], how='left')
weekly['units_sold'] = weekly['units_sold'].fillna(0).astype(int)
weekly = weekly.sort_values(group_columns + ['week_start']).reset_index(drop=True)

print(f'Product-branch combinations: {len(product_branches)}')
print(f'Weeks in the timeline: {len(all_weeks)}')
print(f'Weekly rows, including zero-sales weeks: {len(weekly):,}')
weekly.head()

Product-branch combinations: 20
Weeks in the timeline: 53
Weekly rows, including zero-sales weeks: 1,060


,store_id,store_name,product_id,product_name,week_start,units_sold
0,S001,Sample CBD Mini-Mart,P001,Maize flour 2kg,2024-12-30,4
1,S001,Sample CBD Mini-Mart,P001,Maize flour 2kg,2025-01-06,3
2,S001,Sample CBD Mini-Mart,P001,Maize flour 2kg,2025-01-13,4
3,S001,Sample CBD Mini-Mart,P001,Maize flour 2kg,2025-01-20,1
4,S001,Sample CBD Mini-Mart,P001,Maize flour 2kg,2025-01-27,6


## 3. Create the time-based validation split

The first 80% of weeks are history. The latest 20% are kept aside to check forecasts on weeks the model did not use as training history. For each validation week, both methods only use actual sales from earlier weeks.

In [8]:
week_list = weekly['week_start'].drop_duplicates().sort_values().tolist()
split_number = int(len(week_list) * 0.80)
training_weeks = week_list[:split_number]
validation_weeks = week_list[split_number:]

print(f'History weeks: {len(training_weeks)}')
print(f'Validation weeks: {len(validation_weeks)}')
print(f'Validation period: {validation_weeks[0]:%d %b %Y} to {validation_weeks[-1]:%d %b %Y}')

# For each product and branch, these columns contain past weeks only.
for number_of_weeks_ago in range(1, 5):
    weekly[f'week_{number_of_weeks_ago}_ago'] = (
        weekly.groupby(group_columns)['units_sold'].shift(number_of_weeks_ago)
    )

weekly['baseline_forecast'] = weekly['week_1_ago']
past_week_columns = [f'week_{number}_ago' for number in range(1, 5)]
weekly['four_week_average_forecast'] = weekly[past_week_columns].mean(axis=1)

validation = weekly[weekly['week_start'].isin(validation_weeks)].copy()
validation = validation.dropna(subset=['baseline_forecast', 'four_week_average_forecast'])
validation[['week_start', 'store_name', 'product_name', 'units_sold', 'baseline_forecast', 'four_week_average_forecast']].head()

History weeks: 42
Validation weeks: 11
Validation period: 20 Oct 2025 to 29 Dec 2025


,week_start,store_name,product_name,units_sold,baseline_forecast,four_week_average_forecast
42,2025-10-20,Sample CBD Mini-Mart,Maize flour 2kg,2,6.0,4.75
43,2025-10-27,Sample CBD Mini-Mart,Maize flour 2kg,4,2.0,4.50
44,2025-11-03,Sample CBD Mini-Mart,Maize flour 2kg,2,4.0,4.00
45,2025-11-10,Sample CBD Mini-Mart,Maize flour 2kg,4,2.0,3.50
46,2025-11-17,Sample CBD Mini-Mart,Maize flour 2kg,3,4.0,3.00


## 4. Compare forecasts with actual sales

- **MAE** is the average number of units the forecast missed by. Lower is better.
- **WAPE** is the total absolute miss divided by total actual units. It gives a percentage view across all products. We use it instead of MAPE because some weeks have zero sales, and MAPE cannot divide by zero.

We also show forecast bias: positive means the forecast was higher than actual sales on average; negative means it was lower.

In [9]:
validation['baseline_error'] = validation['baseline_forecast'] - validation['units_sold']
validation['average_error'] = validation['four_week_average_forecast'] - validation['units_sold']
validation['baseline_absolute_error'] = validation['baseline_error'].abs()
validation['average_absolute_error'] = validation['average_error'].abs()

actual_total = validation['units_sold'].sum()
baseline_mae = validation['baseline_absolute_error'].mean()
average_mae = validation['average_absolute_error'].mean()
baseline_wape = validation['baseline_absolute_error'].sum() / actual_total * 100
average_wape = validation['average_absolute_error'].sum() / actual_total * 100
baseline_bias = validation['baseline_error'].mean()
average_bias = validation['average_error'].mean()

evaluation = pd.DataFrame([
    {'forecast': 'Baseline: last week', 'MAE_units': baseline_mae, 'WAPE_pct': baseline_wape, 'bias_units': baseline_bias},
    {'forecast': 'Model: previous 4-week average', 'MAE_units': average_mae, 'WAPE_pct': average_wape, 'bias_units': average_bias},
])
evaluation = evaluation.round(2)
evaluation

,forecast,MAE_units,WAPE_pct,bias_units
0,Baseline: last week,2.20,104.53,0.15
1,Model: previous 4-week average,1.73,82.00,0.12


## 5. What the model gets wrong

The table below shows validation weeks with the largest misses. This helps us see examples of demand spikes the moving average did not catch and quiet weeks where it forecast more than was sold.

In [10]:
validation['miss_direction'] = validation['average_error'].apply(
    lambda error: 'over-forecast' if error > 0 else ('under-forecast' if error < 0 else 'exact')
)
largest_misses = validation.sort_values('average_absolute_error', ascending=False)[
    ['week_start', 'store_name', 'product_name', 'units_sold', 'four_week_average_forecast', 'miss_direction', 'average_absolute_error']
].head(5).copy()
largest_misses[['four_week_average_forecast', 'average_absolute_error']] = largest_misses[['four_week_average_forecast', 'average_absolute_error']].round(2)
largest_misses

,week_start,store_name,product_name,units_sold,four_week_average_forecast,miss_direction,average_absolute_error
581,2025-12-22,Sample Ruiru Mini-Mart,Maize flour 2kg,10,3.00,under-forecast,7.00
574,2025-11-03,Sample Ruiru Mini-Mart,Maize flour 2kg,10,3.25,under-forecast,6.75
891,2025-10-27,Sample Ruiru Mini-Mart,Rice 2kg,7,1.25,under-forecast,5.75
686,2025-12-15,Sample Ruiru Mini-Mart,Bread loaf,0,5.25,over-forecast,5.25
679,2025-10-27,Sample Ruiru Mini-Mart,Bread loaf,7,1.75,under-forecast,5.25


## 6. Conclusion and saved results

The four-week average is useful only if it beats the baseline on the held-out weeks. Even if it does, its misses and the short synthetic history matter.

In [11]:
if average_mae < baseline_mae:
    conclusion = (
        f'The four-week average performed better than the last-week baseline on the validation weeks. '
        f'It missed by {average_mae:.2f} units per product-branch-week on average, compared with {baseline_mae:.2f} for the baseline.'
    )
else:
    conclusion = (
        f'The four-week average did not beat the last-week baseline on the validation weeks. '
        f'It missed by {average_mae:.2f} units per product-branch-week on average, compared with {baseline_mae:.2f} for the baseline.'
    )
print(conclusion)
print(f'Model WAPE: {average_wape:.1f}%. Lower is better; this percentage also shows remaining error.')

# After evaluation, use the full available history for a next-week forecast.
last_observed_week = weekly['week_start'].max()
forecast_week_start = last_observed_week + pd.Timedelta(days=7)
next_week_forecast = weekly.loc[
    weekly['week_start'].eq(last_observed_week),
    group_columns + ['four_week_average_forecast'],
].copy()
next_week_forecast = next_week_forecast.rename(
    columns={'four_week_average_forecast': 'forecast_units'}
)
next_week_forecast['forecast_week_start'] = forecast_week_start
next_week_forecast['forecast_units'] = next_week_forecast['forecast_units'].round(2)
next_week_forecast.to_csv(REPORTS_DIR / 'next_week_forecast.csv', index=False)
print(f'Next-week forecast saved for week starting {forecast_week_start:%d %b %Y}.')

worst_misses_text = largest_misses.to_string(index=False)
report_lines = [
    '# Demand Forecast Evaluation',
    '',
    'This forecast uses synthetic weekly sales data. It is a learning example, not an order recommendation for a real shop.',
    '',
    '## Forecast setup',
    '',
    f'Training/history: {len(training_weeks)} earlier weeks. Validation: {len(validation_weeks)} later weeks, from {validation_weeks[0]:%d %b %Y} to {validation_weeks[-1]:%d %b %Y}.',
    'The baseline uses the previous week. The model uses the average of the previous four weeks. Both use past actual sales only for each one-week-ahead forecast.',
    '',
    '## Validation metrics',
    '',
    '| Forecast | MAE (units) | WAPE | Bias (units) |',
    '| --- | ---: | ---: | ---: |',
    f'| Last-week baseline | {baseline_mae:.2f} | {baseline_wape:.1f}% | {baseline_bias:.2f} |',
    f'| Four-week average model | {average_mae:.2f} | {average_wape:.1f}% | {average_bias:.2f} |',
    '',
    '## Conclusion',
    '',
    conclusion,
    '',
    '## Next-week forecast',
    '',
    f'The four-week average forecast for the week starting {forecast_week_start:%d %b %Y} was created using all available history.',
    'The product-by-branch values are saved in reports/next_week_forecast.csv.',
    '',
    '## Largest model misses',
    '',
    '```text',
    worst_misses_text,
    '```',
    '',
    'The model smooths demand, so it can miss sudden spikes and can forecast sales in a week that turns out quiet. The validation covers only the final part of one synthetic year, so performance may differ in another year or a real store.'
]

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
(REPORTS_DIR / 'evaluation.md').write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
validation[['week_start', 'store_name', 'product_name', 'units_sold', 'baseline_forecast', 'four_week_average_forecast', 'baseline_absolute_error', 'average_absolute_error']].to_csv(
    REPORTS_DIR / 'validation_predictions.csv', index=False
)
print('Saved the evaluation report, validation predictions, and next-week forecast in reports/')

The four-week average performed better than the last-week baseline on the validation weeks. It missed by 1.73 units per product-branch-week on average, compared with 2.20 for the baseline.
Model WAPE: 82.0%. Lower is better; this percentage also shows remaining error.
Next-week forecast saved for week starting 05 Jan 2026.
Saved the evaluation report, validation predictions, and next-week forecast in reports/
